# Chapter 3 — Feature Selection

This notebook corresponds to Week 3 of *Applied Machine Learning* and uses the UCI Forest Fires dataset. Place the book's `Data-Week3.csv` file in `data/raw/` at the repository root.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

data_path = Path('../../data/raw/Data-Week3.csv')
data = pd.read_csv(data_path)
data.head(3)

## Correlation heat map

Compatibility note: current pandas versions require non-numeric columns such as `month` and `day` to be excluded explicitly. Older pandas versions did this implicitly. This does not change the numeric correlation method used in the book.


In [ ]:
numeric_data = data.select_dtypes(include='number')
correlations = numeric_data.corr()
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111)
cax = ax.matshow(correlations, vmin=-1, vmax=1)
fig.colorbar(cax)
ticks = np.arange(0, len(numeric_data.columns), 1)
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_xticklabels(numeric_data.columns, rotation=90)
ax.set_yticklabels(numeric_data.columns)
plt.show()

The book notes high correlation between temperature and ISI, and between DMC and DC.

## Scatter-plot matrix


In [ ]:
pd.plotting.scatter_matrix(data)
plt.show()

## Prepare predictors and target

The positional column selections are retained from the book and require the same prepared CSV column order.


In [ ]:
X = data.iloc[:, 2:9]
Y = data.iloc[:, 0]

## Univariate selection


In [ ]:
from numpy import set_printoptions
from sklearn.feature_selection import SelectKBest, chi2

test = SelectKBest(score_func=chi2, k=4)
fit = test.fit(X, Y)
set_printoptions(precision=2)
print(fit.scores_)
featured_data = fit.transform(X)
print('\nFeatured data:\n', featured_data[0:4])

Book interpretation: selected features are DMC, DC, temperature, and ISI.

## Recursive feature elimination


In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
rfe = RFE(model, n_features_to_select=4)
fit = rfe.fit(X, Y)
print(fit.support_)
print(fit.ranking_)

Book interpretation: selected features are FFMC, DMC, temperature, and RH.

## Extra Trees feature importance


In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

model = ExtraTreesClassifier()
model.fit(X, Y)
print(model.feature_importances_)

Book interpretation: selected features are FFMC, DMC, and DC. Because the original model has no fixed random seed, feature importances can vary between runs.
